# The error is not where it happened
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/demos/week5/Error_At_Next_Sync_Demo_Colab.ipynb)

**Dataflow Architectures for AI, Week 5, Module 1: Debugging compiled programs**  
**Author:** Dr. Kaoutar El Maghraoui

The host queues work for the GPU and returns immediately, so Python runs ahead of the device. When a kernel faults, the host finds out only when it reaches a call that waits for the device, such as `.item()`. The traceback points there, not at the operation that failed. On CUDA the error is also **sticky**: every later CUDA call in the same process reports it again.

The lecture's fix is to make launches synchronous while you localize the fault, with `CUDA_LAUNCH_BLOCKING=1`. Compilation adds a second problem: one kernel covers many operations, so even a synchronous error names a compiled region, not a line. And a `print` inside a compiled function changes the program you are debugging.

In this notebook you plant an out-of-range index, run it four ways, and read where each traceback points.

**What you will do**
1. Write a small script with a bad embedding index.
2. Run it in eager mode and see the error reported at `.item()`.
3. Run it again with `CUDA_LAUNCH_BLOCKING=1` and see the error at the real line.
4. Run the compiled version and see what the traceback names.
5. See that a `print` inside a compiled function splits the graph.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Each run is a separate Python process, so the sticky error never breaks this notebook. The notebook takes about two minutes.

## 1. The script
The embedding table has 100 rows. The input contains the index 150, which is out of range. The embedding kernel detects it on the GPU and raises a device-side assert. The script then does more work, and only at the end asks for a value on the host with `.item()`.

The comments in capitals mark the lines to watch: `FAULT` is where the bad index is read, `CALL` is where the work is launched, and `SYNC` is where the host waits for the GPU. The helper in the next section prints the source line of every traceback frame, so you can match them.

In [ ]:
%%writefile /content/sync_demo.py
import sys
import torch

mode = sys.argv[1]                                   # "eager" or "compiled"
emb = torch.nn.Embedding(100, 64).cuda()             # a table with 100 rows


def step(ids):
    h = emb(ids)                                     # FAULT: reads row 150, which does not exist
    h = torch.tanh(h * 2.0)
    return h.sum(-1)


ids = torch.tensor([[1, 2, 150, 4]], device="cuda")
fn = torch.compile(step) if mode == "compiled" else step
try:
    out = fn(ids)                                    # CALL: launches the work and returns at once
    print("host: step() returned, the GPU may still be running", flush=True)
    total = out.sum().item()                         # SYNC: the host waits for the GPU here
except Exception as err:
    print("host: error raised:", type(err).__name__, flush=True)
    try:
        torch.zeros(1, device="cuda").item()         # an unrelated CUDA call
        print("host: an unrelated CUDA call still works", flush=True)
    except Exception as err2:
        print("host: an unrelated CUDA call fails too:", type(err2).__name__, flush=True)
    raise

## 2. A helper that runs the script and shows the traceback
The helper runs the script in a new process and prints three things: what the script printed, the source line of each traceback frame inside `sync_demo.py`, and the last error line.

In [ ]:
import os
import re
import subprocess


def run(mode, **env_vars):
    env = dict(os.environ, **env_vars)
    p = subprocess.run(["python", "/content/sync_demo.py", mode], capture_output=True, text=True, env=env)
    for line in p.stdout.splitlines():
        print("  " + line)
    source = open("/content/sync_demo.py").read().splitlines()
    frames = re.findall(r'File "/content/sync_demo.py", line (\d+)', p.stderr)
    print("  traceback frames in sync_demo.py:")
    for n in frames:
        print(f"    line {n}: {source[int(n) - 1].strip()}")
    if not frames:
        print("    none")
    errors = [l for l in p.stderr.splitlines() if "Error" in l or "error" in l]
    if errors:
        print("  last error line:", errors[-1][:160])
    return p

## 3. Eager mode, asynchronous launches

In [ ]:
print("eager, default settings")
p_eager = run("eager")

**Reading the output**
- The traceback does not point at the `FAULT` line, where the bad index was read. It points at a later line: either the `tanh` line, the next operation after the fault, or the `SYNC` line. The error is reported by the first CUDA call that checks for errors after the GPU has hit the fault, so which line you see depends on timing. If the host message `step() returned` was printed, the host had already moved on before the error arrived.
- The unrelated `torch.zeros(1)` call fails too. The CUDA context is now in an error state, and only a new process recovers. In a long notebook this is why one bad index makes every later cell fail.

## 4. Eager mode, synchronous launches
With `CUDA_LAUNCH_BLOCKING=1`, every kernel launch waits for the kernel to finish. Errors appear at the operation that caused them. It is slow, so use it only while you localize a fault.

In [ ]:
print("eager, CUDA_LAUNCH_BLOCKING=1")
p_blocking = run("eager", CUDA_LAUNCH_BLOCKING="1")

**Reading the output**
- The traceback now ends at the `FAULT` line, the embedding lookup. That is the real fault, and it is the same on every run.
- The error was raised inside `step()`, before it returned, so the host message is missing.

## 5. Compiled, synchronous launches
Under `torch.compile`, Inductor fuses the embedding lookup, the multiply, the tanh and the sum into generated kernels, and it adds its own bounds check to the generated code. The fault is still found, but the traceback ends in the compiled function, not in one source line.

In [ ]:
print("compiled, CUDA_LAUNCH_BLOCKING=1")
p_compiled = run("compiled", CUDA_LAUNCH_BLOCKING="1")
generated = [l.strip() for l in p_compiled.stderr.splitlines()
             if "torchinductor" in l or "device_assert" in l or "index out of bounds" in l]
print("  lines that point into generated code:")
for line in generated[:2]:
    print("    " + line[:150])

**Reading the output**
- The frames in `sync_demo.py` point at the `CALL` line and at `def step(ids):`, the start of the compiled region. They do not point at the `FAULT` line: the three lines of `step` became one compiled region.
- The remaining frames are in files under `/tmp/torchinductor_...`, which Inductor generated. The message may say `index out of bounds`: that check was written by the compiler, not by you.
- To get from a kernel back to a source line, use the provenance tools from Module 3. The provenance demo in this folder shows them.

## 6. A print changes the graph
A `print` inside a compiled function is a graph break. Dynamo splits the graph in two, writes the intermediate to memory so Python can print it, and compiles each half separately. You are now debugging a different program from the one that failed. `torch._dynamo.explain` reports the graphs and the breaks without running the compiled code.

In [ ]:
import torch


def plain(x):
    y = x * 2 + 1
    return torch.relu(y)


def with_print(x):
    y = x * 2 + 1
    print("y mean:", y.mean())         # debugging print inside the function
    return torch.relu(y)


x = torch.randn(1024, device="cuda")
for fn in (plain, with_print):
    torch._dynamo.reset()
    info = torch._dynamo.explain(fn)(x)
    print(f"{fn.__name__:10}: {info.graph_count} graph(s), {info.graph_break_count} break(s)")
    for reason in info.break_reasons:
        print("    break reason:", str(reason.reason).splitlines()[0][:120])

**Reading the output**
- `plain` is one graph with no breaks. Inductor can fuse the multiply, add and ReLU into one kernel.
- `with_print` is two graphs and one break. The intermediate `y` is now written to memory between them, so the fused kernel no longer exists, and at low precision the split version can round differently.
- The lecture's workaround is to register `print` as a reorderable logging function, which moves the call after the graph runs instead of splitting it.

## Try this
1. In the script, move `.item()` to directly after `fn(ids)`. Where does the eager traceback point now, without `CUDA_LAUNCH_BLOCKING`?
2. Run `run("compiled")` without `CUDA_LAUNCH_BLOCKING`. Which line does the traceback point at?
3. In section 6, replace `print` with `torch._dynamo.config.reorderable_logging_functions.add(print)` before compiling, and check the graph count again.

## Summary
- GPU launches are asynchronous, so a device fault is reported at the next synchronization, such as `.item()`.
- On CUDA a device error is sticky: every later call in the process fails, and only a restart recovers.
- `CUDA_LAUNCH_BLOCKING=1` makes the traceback point at the operation that faulted, in eager mode.
- In compiled code, one kernel covers many operations, so the traceback names the compiled region. Provenance tracking maps it back to source lines.
- A `print` inside a compiled function splits the graph and changes the program.

Lecture: Week 5, Module 1, slides 8 to 11 (what a compiled target removes, error at next sync, print changes the graph, one kernel many ops).